# 24 - Daughter grid at early transitions

The final scans should reach a_t = 10⁻³, but there the 4-parameter scan's 201 daughter bins fail: at a_t = 10⁻³,
f̃ = 0.9 they fake Δχ² ≈ 4 from 10¹⁵ GeV up, and 801 bins were not shown to be converged at 10¹⁶ GeV (nb22,
section 5; `accdm_l_max` does not matter). How many bins does each (m, a_t) need?

Ladder: 201, 401, 801 and 1601 daughter bins (1601 the reference) at 10¹⁴–10¹⁸ GeV, f̃ = 0.5 and 0.9, for κ = 12.1 at
a_t = 10⁻³, 3·10⁻³, 10⁻², and for κ = 1 and 20 at a_t = 10⁻³.

1. Signal (Δχ² against f̃ → 0) and error (Δχ² against 1601 bins) per bin count.
2. max |ΔP_cb/P_cb| against 1601 bins.
3. The bins each point needs for an error below 0.1 (and 0.01), and their cost.

**Result.** The bins needed grow fast below a_t = 10⁻², independent of mass:
- **a_t = 10⁻²:** 201 bins are converged (Δχ² ≤ 0.008 against 1601).
- **a_t = 3·10⁻³:** 201 bins err by ≤ 0.07, 401 bins by ≤ 0.007.
- **a_t = 10⁻³:** 201 bins err by 0.3–3.5 and 401 by 0.03–0.19; 801 bins reach ≤ 0.014 (κ = 1, 12.1, 20). The error is
  nearly mass-independent while the signal falls to ~10⁻³ above 10¹⁵ GeV, which is why 201 bins faked Δχ² ≈ 4 there
  (nb22). Each doubling cuts the error 5–10×, so 1601 is converged to ~0.002.
- A rule N = 201·max(1, 10⁻²/a_t)^0.6 (201, 410, 800 at 10⁻², 3·10⁻³, 10⁻³) meets 0.01 everywhere tested. Cost per run
  is 1, 2.0, 4.5, 10.5× for 201, 401, 801, 1601 bins; with a_t log-flat on [10⁻³, 1] the rule averages ~1.5× of 201,
  against 4.5× for a fixed 801.
- **P_cb:** 201 bins leave max |ΔP_cb/P_cb| ≤ 2.6·10⁻³ (7·10⁻³ at κ = 20, a_t = 10⁻³), 801 bins ≤ 5·10⁻⁴; matters for
  weak lensing, not for Planck + BAO.
- **κ at full resolution:** at a_t = 10⁻³ (same z₅₀ for steep transitions), κ = 12.1 and 20 give the same signal to ≤ 1%
  wherever Δχ² ≳ 0.3 (1601 bins), so κ ≳ 4 acting only through z₅₀ is not a grid effect; κ = 1 is 3–10× stronger.

Settings: `CHAIN_SETTINGS_OCT07`, `accdm_l_max` 17, default kick. Δχ²: `accdm_nb.chi2_parts`. Runs come from kruk:
`cluster_scripts/gen_nb24_cache.py` defines the grid (imported here) and writes `accDM_scans/nb24_cache/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from accdm_nb import RunCache, chi2_parts, ok, show, status
from cluster_scripts.gen_nb24_cache import BINS, MASSES, F_TILDES, TRANSITIONS, NULL, params, jobs

SCANS = Path('accDM_scans/nb24_cache')
missing = [tag for tag, _ in jobs() if not (SCANS/(tag + '.pkl')).exists()]
assert not missing, '{} jobs missing, e.g. {}: run gen_nb24_cache.py on kruk'.format(len(missing), missing[:3])
cache = RunCache('24_cache.pkl', also=sorted(SCANS.glob('*.pkl')))


def runs(ps):
    # cached results of ps; never runs CLASS
    out = [cache._get(cache.key(p)) for p in ps]
    assert all(o is not None for o in out), 'a run is missing from nb24_cache'
    return out


null = runs([NULL])[0]
print('{} runs read; failed:'.format(len(cache.extra)), [status(o) for o in cache.extra.values() if not ok(o)] or 'none')

201 runs read; failed: none


## 1. Signal and error per bin count

*signal*: Δχ² of the reference (1601 bins) against the null. *Δχ² N*: N bins against the reference.

In [2]:
REF = BINS[-1]
rows = []
for k, at in TRANSITIONS:
    for lm in MASSES:
        for ft in F_TILDES:
            res = dict(zip(BINS, runs([params(lm, ft, k, at, b) for b in BINS])))
            ref = res[REF]
            rows.append({'κ': k, 'a_t': at, 'log10 m': lm, 'f̃': ft, 'signal': chi2_parts(ref, null)['total'],
                         **{'Δχ² {}'.format(b): chi2_parts(res[b], ref)['total'] for b in BINS[:-1]},
                         **{'ΔP {}'.format(b): float(np.max(np.abs(res[b]['pk_cb']/ref['pk_cb'] - 1))) for b in BINS[:-1]},
                         **{'sec {}'.format(b): res[b]['sec'] for b in BINS}})
t = pd.DataFrame(rows).set_index(['κ', 'a_t', 'log10 m', 'f̃'])
show(t[['signal'] + ['Δχ² {}'.format(b) for b in BINS[:-1]]], default='{:.2g}')

signal Δχ² 201  Δχ² 401  Δχ² 801
κ    a_t   log10 m f̃                                    
12.1 0.001 14      0.5       18   0.074    0.031   0.0066
                   0.9       49    0.99     0.19   0.0041
           15      0.5     0.34    0.35    0.065   0.0034
                   0.9        1    0.71     0.17    0.012
           16      0.5    0.005    0.32    0.063   0.0031
                   0.9    0.016     0.7     0.14    0.013
           17      0.5  0.00074    0.32    0.063   0.0027
                   0.9   0.0046    0.69     0.14    0.013
           18      0.5   0.0007    0.32    0.064    0.003
                   0.9   0.0037    0.69     0.14    0.012
     0.003 14      0.5       84    0.01  0.00029  4.9e-05
                   0.9    2e+02   0.069   0.0071  4.9e-05
           15      0.5      2.3  0.0072   0.0004  0.00013
                   0.9      6.8   0.055    0.003  0.00032
           16      0.5    0.034  0.0076  0.00039  0.00011
                   0.9     0.11   0.052   0.0027  3.6e-05
           17      0.5  0.00038  0.0073  0.00037   0.0001
                   0.9   0.0013   0.052   0.0028  0.00013
           18      0.5  6.6e-06  0.0075  0.00039  0.00011
                   0.9  3.9e-05   0.069   0.0029  0.00019
     0.010 14      0.5    3e+02  0.0026  0.00075  0.00011
                   0.9  6.1e+02  0.0083   0.0043  0.00016
           15      0.5       15  0.0027  0.00081  0.00015
                   0.9       42  0.0064   0.0053   0.0004
           16      0.5     0.29  0.0027  0.00084  6.5e-05
                   0.9     0.89  0.0064   0.0055  0.00047
           17      0.5   0.0036  0.0027  0.00083  0.00014
                   0.9    0.012  0.0063   0.0052  0.00038
           18      0.5  6.1e-05  0.0027  0.00079  4.5e-05
                   0.9  0.00028  0.0062   0.0051  0.00038
1.0  0.001 14      0.5       49    0.14    0.031   0.0035
                   0.9  1.3e+02    0.37     0.12    0.014
           15      0.5        2    0.14    0.036   0.0034
                   0.9        6    0.37     0.11   0.0099
           16      0.5    0.048    0.14    0.036   0.0034
                   0.9     0.15    0.37     0.11     0.01
           17      0.5   0.0075    0.14    0.037   0.0035
                   0.9    0.021    0.37     0.11   0.0097
           18      0.5   0.0066    0.14    0.036   0.0035
                   0.9    0.019    0.37     0.11   0.0099
20.0 0.001 14      0.5       18    0.26    0.012    0.001
                   0.9       48    0.94     0.07   0.0026
           15      0.5     0.34    0.21   0.0075   0.0012
                   0.9        1     3.5     0.15    0.004
           16      0.5    0.005     0.2   0.0067   0.0012
                   0.9    0.015     3.4     0.15   0.0016
           17      0.5  0.00089     0.2   0.0056   0.0027
                   0.9   0.0025     3.4     0.14   0.0016
           18      0.5  0.00058     0.2   0.0065   0.0015
                   0.9    0.002     3.4     0.14   0.0017

## 2. P_cb

max |ΔP_cb/P_cb| over k ≤ 0.9/Mpc against the reference: what P(k)-based likelihoods (weak lensing) see.

In [3]:
show(t[['ΔP {}'.format(b) for b in BINS[:-1]]], default='{:.1e}')

ΔP 201   ΔP 401   ΔP 801
κ    a_t   log10 m f̃                            
12.1 0.001 14      0.5  2.1e-04  1.7e-04  6.3e-05
                   0.9  2.4e-03  1.3e-03  4.5e-04
           15      0.5  1.8e-04  1.5e-04  5.1e-05
                   0.9  1.7e-03  9.3e-04  3.2e-04
           16      0.5  1.3e-04  3.8e-05  1.2e-05
                   0.9  3.4e-04  1.2e-04  4.0e-05
           17      0.5  1.3e-04  3.9e-05  1.3e-05
                   0.9  3.5e-04  1.2e-04  4.1e-05
           18      0.5  1.3e-04  3.9e-05  1.3e-05
                   0.9  3.5e-04  1.2e-04  4.1e-05
     0.003 14      0.5  4.8e-04  2.0e-04  7.4e-05
                   0.9  2.6e-03  1.1e-03  3.8e-04
           15      0.5  4.6e-04  1.9e-04  6.7e-05
                   0.9  2.5e-03  1.0e-03  3.6e-04
           16      0.5  1.6e-04  6.8e-05  2.4e-05
                   0.9  4.7e-04  2.0e-04  6.9e-05
           17      0.5  3.2e-05  1.2e-05  3.7e-06
                   0.9  8.7e-05  3.4e-05  1.1e-05
           18      0.5  3.2e-05  1.2e-05  3.8e-06
                   0.9  8.7e-05  3.4e-05  1.1e-05
     0.010 14      0.5  4.9e-04  2.2e-04  7.4e-05
                   0.9  2.2e-03  9.8e-04  3.2e-04
           15      0.5  4.7e-04  2.1e-04  6.6e-05
                   0.9  2.1e-03  9.5e-04  3.0e-04
           16      0.5  4.1e-04  1.9e-04  6.0e-05
                   0.9  1.8e-03  8.2e-04  2.6e-04
           17      0.5  5.9e-05  2.8e-05  8.9e-06
                   0.9  1.6e-04  7.4e-05  2.3e-05
           18      0.5  5.1e-06  3.6e-06  1.1e-06
                   0.9  1.6e-05  9.5e-06  2.9e-06
1.0  0.001 14      0.5  3.7e-04  2.1e-04  8.4e-05
                   0.9  1.5e-03  6.5e-04  2.4e-04
           15      0.5  2.8e-04  1.3e-04  6.4e-05
                   0.9  1.0e-03  4.5e-04  1.9e-04
           16      0.5  1.5e-04  8.6e-05  3.6e-05
                   0.9  2.2e-04  1.2e-04  7.1e-05
           17      0.5  1.5e-04  8.5e-05  3.0e-05
                   0.9  2.1e-04  1.2e-04  4.4e-05
           18      0.5  1.5e-04  8.5e-05  3.0e-05
                   0.9  2.1e-04  1.2e-04  4.2e-05
20.0 0.001 14      0.5  3.3e-03  2.0e-04  6.4e-05
                   0.9  7.2e-03  1.4e-03  4.9e-04
           15      0.5  1.7e-03  1.7e-04  5.3e-05
                   0.9  3.4e-03  1.0e-03  3.4e-04
           16      0.5  8.3e-04  3.1e-05  1.1e-05
                   0.9  1.5e-03  1.0e-04  3.7e-05
           17      0.5  7.9e-04  3.3e-05  1.3e-05
                   0.9  1.5e-03  1.1e-04  3.8e-05
           18      0.5  7.9e-04  3.3e-05  1.3e-05
                   0.9  1.5e-03  1.1e-04  3.8e-05

## 3. Bins needed, and their cost

Smallest bin count with Δχ² against the reference below 0.1 (and 0.01) at both f̃, per (κ, a_t, m); *1601?* means
only the reference itself, i.e. convergence is not shown. Cost: wall time of one run on a 4-thread node, relative to
201 bins.

In [4]:
def needed(g, tol):
    for b in BINS[:-1]:
        if (g['Δχ² {}'.format(b)] < tol).all():
            return str(b)
    return '1601?'


need = t.groupby(level=['κ', 'a_t', 'log10 m']).apply(lambda g: pd.Series({'for 0.1': needed(g, 0.1), 'for 0.01': needed(g, 0.01)}))
show(need, default='{}')
cost = pd.DataFrame({'bins': BINS, 'median sec': [t['sec {}'.format(b)].median() for b in BINS]})
cost['× 201'] = cost['median sec']/cost['median sec'].iloc[0]
show(cost.set_index('bins'), {'median sec': '{:.0f}', '× 201': '{:.1f}'})

for 0.1 for 0.01
κ    a_t   log10 m                 
1.0  0.001 14          801    1601?
           15          801      801
           16          801    1601?
           17          801      801
           18          801      801
12.1 0.001 14          801      801
           15          801    1601?
           16          801    1601?
           17          801    1601?
           18          801    1601?
     0.003 14          201      401
           15          201      401
           16          201      401
           17          201      401
           18          201      401
     0.010 14          201      201
           15          201      201
           16          201      201
           17          201      201
           18          201      201
20.0 0.001 14          401      801
           15          801      801
           16          801      801
           17          801      801
           18          801      801

,median sec,× 201
bins,,
201,54,1.0
401,109,2.0
801,244,4.5
1601,565,10.5
